In [19]:
import pandas as pd
import numpy as np

import yaml
from pathlib import Path

In [20]:
config_path = Path.cwd().parent / "config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

root = Path(config['project_root'])

In [21]:
df = pd.read_parquet("../data/processed/air_quality_weather.parquet")
df.head()

,Date,station,Town,Province,Latitude,Longitude,NO2,PM10,PM2.5,SO2,Temperature,Humidity,Precipitation,wind_u,wind_v,WindSpeed,WindDirection
0,2015-01-01,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,47.0,25.0,28.0,9.0,6.4,85,0.0,-7.980512,0.558052,8.0,176
1,2015-01-02,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,56.0,24.0,18.0,8.0,8.3,81,0.0,-7.929958,-4.040515,8.9,207
2,2015-01-03,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,48.0,33.0,21.0,8.0,8.9,80,0.0,-10.648977,-7.456494,13.0,215
3,2015-01-04,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,43.0,31.0,23.0,7.0,9.6,88,0.0,-6.761170,-5.475087,8.7,219
4,2015-01-05,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,29.0,18.0,11.0,5.0,9.0,85,0.0,-13.688828,2.413710,13.9,170


## Temporal Feature Engineering

In [22]:
df["year"] = df["Date"].dt.year
df["month"] = df["Date"].dt.month
df["day"] = df["Date"].dt.day
df["day_of_year"] = df["Date"].dt.dayofyear
df["week_of_year"] = df["Date"].dt.isocalendar().week
df["day_of_week"] = df["Date"].dt.dayofweek
df['is_weekend'] = df['Date'].dt.weekday >= 5
df['season'] = df['month'].map({12:0,1:0,2:0, 3:1,4:1,5:1, 6:2,7:2,8:2, 9:3,10:3,11:3})
df.head()

,Date,station,Town,Province,Latitude,Longitude,NO2,PM10,PM2.5,SO2,...,WindSpeed,WindDirection,year,month,day,day_of_year,week_of_year,day_of_week,is_weekend,season
0,2015-01-01,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,47.0,25.0,28.0,9.0,...,8.0,176,2015,1,1,1,1,3,False,0
1,2015-01-02,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,56.0,24.0,18.0,8.0,...,8.9,207,2015,1,2,2,1,4,False,0
2,2015-01-03,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,48.0,33.0,21.0,8.0,...,13.0,215,2015,1,3,3,1,5,True,0
3,2015-01-04,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,43.0,31.0,23.0,7.0,...,8.7,219,2015,1,4,4,1,6,True,0
4,2015-01-05,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,29.0,18.0,11.0,5.0,...,13.9,170,2015,1,5,5,2,0,False,0


In [23]:
# Build target columns if missing
df = df.sort_values(["station", "Date"]).copy()

for pollutant in ["PM2.5", "PM10", "NO2", "SO2"]:
    p = pollutant.replace(".", "")
    tcol = f"target_{p}"
    if tcol not in df.columns:
        df[tcol] = df.groupby("station")[pollutant].shift(-1)

print("Target columns now:", [c for c in df.columns if c.startswith("target_")])
df.head()

Target columns now: ['target_PM25', 'target_PM10', 'target_NO2', 'target_SO2']


,Date,station,Town,Province,Latitude,Longitude,NO2,PM10,PM2.5,SO2,...,day,day_of_year,week_of_year,day_of_week,is_weekend,season,target_PM25,target_PM10,target_NO2,target_SO2
0,2015-01-01,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,47.0,25.0,28.0,9.0,...,1,1,1,3,False,0,18.0,24.0,56.0,8.0
1,2015-01-02,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,56.0,24.0,18.0,8.0,...,2,2,1,4,False,0,21.0,33.0,48.0,8.0
2,2015-01-03,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,48.0,33.0,21.0,8.0,...,3,3,1,5,True,0,23.0,31.0,43.0,7.0
3,2015-01-04,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,43.0,31.0,23.0,7.0,...,4,4,1,6,True,0,11.0,18.0,29.0,5.0
4,2015-01-05,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,29.0,18.0,11.0,5.0,...,5,5,2,0,False,0,24.0,30.0,47.0,9.0


In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 29008 entries, 0 to 29007
Data columns (total 29 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Date           29008 non-null  datetime64[us]
 1   station        29008 non-null  str           
 2   Town           29008 non-null  str           
 3   Province       29008 non-null  str           
 4   Latitude       29008 non-null  float64       
 5   Longitude      29008 non-null  float64       
 6   NO2            29008 non-null  float64       
 7   PM10           29008 non-null  float64       
 8   PM2.5          29008 non-null  float64       
 9   SO2            29008 non-null  float64       
 10  Temperature    29008 non-null  float64       
 11  Humidity       29008 non-null  int64         
 12  Precipitation  29008 non-null  float64       
 13  wind_u         29008 non-null  float64       
 14  wind_v         29008 non-null  float64       
 15  WindSpeed      29008 non-null 

## Lag Features

In [36]:
lags = [1,3,7,30,90,365]

for lag in lags:
    df[f"PM25_lag_{lag}"] = (
        df.groupby("station")["PM2.5"]
        .shift(lag)
    )

In [37]:
for lag in lags:
    df[f"PM10_lag_{lag}"] = (
        df.groupby("station")["PM10"]
        .shift(lag)
    )

In [38]:
for lag in lags:
    df[f"NO2_lag_{lag}"] = (
        df.groupby("station")["NO2"]
        .shift(lag)
    )

In [39]:
for lag in lags:
    df[f"SO2_lag_{lag}"] = (
        df.groupby("station")["SO2"]
        .shift(lag)
    )

## Rolling Features

In [40]:
windows = [7,14,30,90,365]

for w in windows:
    df[f"PM25_roll_mean_{w}"] = (
        df.groupby("station")["PM2.5"]
        .transform(
            lambda x:
            x.shift(1).rolling(w).mean()
        )
    )

In [41]:
windows = [7,14,30,90,365]

for w in windows:
    df[f"PM10_roll_mean_{w}"] = (
        df.groupby("station")["PM10"]
        .transform(
            lambda x:
            x.shift(1).rolling(w).mean()
        )
    )

In [42]:
windows = [7,14,30,90,365]

for w in windows:
    df[f"NO2_roll_mean_{w}"] = (
        df.groupby("station")["NO2"]
        .transform(
            lambda x:
            x.shift(1).rolling(w).mean()
        )
    )

In [43]:
windows = [7,14,30,90,365]

for w in windows:
    df[f"SO2_roll_mean_{w}"] = (
        df.groupby("station")["SO2"]
        .transform(
            lambda x:
            x.shift(1).rolling(w).mean()
        )
    )

### Forecast 

In [44]:
df

,Date,station,Town,Province,Latitude,Longitude,NO2,PM10,PM2.5,SO2,...,NO2_roll_mean_7,NO2_roll_mean_14,NO2_roll_mean_30,NO2_roll_mean_90,NO2_roll_mean_365,SO2_roll_mean_7,SO2_roll_mean_14,SO2_roll_mean_30,SO2_roll_mean_90,SO2_roll_mean_365
0,2015-01-01,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,47.0,25.0,28.0,9.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-01-02,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,56.0,24.0,18.0,8.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-01-03,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,48.0,33.0,21.0,8.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2015-01-04,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,43.0,31.0,23.0,7.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2015-01-05,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,29.0,18.0,11.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29003,2026-05-02,SANTURCE,Santurtzi,Bizkaia,43.333012,-3.042560,16.0,13.0,11.0,3.0,...,13.428571,15.571429,15.933333,15.744444,15.113077,3.000000,3.071429,3.533333,4.388889,4.356164
29004,2026-05-03,SANTURCE,Santurtzi,Bizkaia,43.333012,-3.042560,12.0,9.0,6.0,3.0,...,14.714286,15.785714,16.266667,15.811111,15.110337,3.000000,3.071429,3.533333,4.377778,4.353425
29005,2026-05-04,SANTURCE,Santurtzi,Bizkaia,43.333012,-3.042560,10.0,5.0,3.0,3.0,...,15.428571,15.785714,16.500000,15.855556,15.118556,3.000000,3.071429,3.500000,4.377778,4.353425
29006,2026-05-05,SANTURCE,Santurtzi,Bizkaia,43.333012,-3.042560,14.0,7.0,4.0,4.0,...,15.000000,15.500000,16.266667,15.900000,15.115817,3.000000,3.071429,3.433333,4.377778,4.353425


In [45]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 29008 entries, 0 to 29007
Data columns (total 73 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Date                29008 non-null  datetime64[us]
 1   station             29008 non-null  str           
 2   Town                29008 non-null  str           
 3   Province            29008 non-null  str           
 4   Latitude            29008 non-null  float64       
 5   Longitude           29008 non-null  float64       
 6   NO2                 29008 non-null  float64       
 7   PM10                29008 non-null  float64       
 8   PM2.5               29008 non-null  float64       
 9   SO2                 29008 non-null  float64       
 10  Temperature         29008 non-null  float64       
 11  Humidity            29008 non-null  int64         
 12  Precipitation       29008 non-null  float64       
 13  wind_u              29008 non-null  float64       
 14  w

In [46]:
df.to_parquet("../data/processed/forecasting_dataset.parquet", index=False)

print("✅ Success: forecasting_dataset.parquet saved.")

✅ Success: forecasting_dataset.parquet saved.
